In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. Create preQC_df, do QC offline
2. Load QC; save maybes, mergers, & clean neur figs in the data/units dir
3. Create neur_df with columns for spikes, region, coordinates
4. Merge clusters as needed
5. Inspect & save

### variables

In [14]:
patient = 21
save = False

units_dir = f'../../data/units'
units_all_dir = f'{units_dir}/all/2025{patient}'
units_waves_dir = f'{units_dir}/waves/2025{patient}'
units_clean_dir = f'{units_dir}/clean/2025{patient}'
units_maybes_dir = f'{units_dir}/maybes/2025{patient}'
units_mergers_dir = f'{units_dir}/mergers/2025{patient}'
for dir in [units_all_dir, units_waves_dir, units_clean_dir, units_maybes_dir, units_mergers_dir]:
    os.makedirs(dir, exist_ok=True)

### 1. set up preQC

In [15]:
# save = True

# parse osort figs
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs/5/*'):

    # grab individual CL figs
    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file):
        
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {dest}/')

    # grab WAVES figs to check for mergers
    if 'WAVES' in os.path.basename(file):
        
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {dest}/')

save = False # reset to be safe

### create preQC_df
note: clustID called unitID going forward

In [16]:
# init preQC_df 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# parse units_all_dir
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:]))
    unitIDs.append(int(os.path.basename(file).split('_')[2]))

# create df
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

# save df
preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if not os.path.exists(preQC_df_path):
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)

save = False # reset to be safe
preQC_df

,chanID,unitID
0,193,21
1,193,166
2,193,232
3,193,268
4,193,331
...,...,...
156,223,2492
157,224,1009
158,224,3097
159,224,3104


### 2. load QC; save maybes, clean, mergers, and possible_neurs (clean+merger) neur figs in the data/units dir

In [17]:
save=True

QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')
maybes_df = QC_df[~QC_df['notes'].isna()].copy().reset_index(drop=True)
clean_df = QC_df[QC_df['keep'] == 1].copy().reset_index(drop=True)
mergers_df = QC_df[~QC_df['merge_cluster'].isna()].copy().reset_index(drop=True)

# clean + mergers
possible_neurs_df = pd.concat([clean_df, mergers_df]).drop_duplicates().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    chanID = int(os.path.basename(unit_file).split('_')[0][1:])
    unitID = int(os.path.basename(unit_file).split('_')[2])
    
    if ((maybes_df['chanID'] == chanID) & (maybes_df['unitID'] == unitID)).any():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

    if ((mergers_df['chanID'] == chanID) & (mergers_df['unitID'] == unitID)).any():
        dest = f'{units_mergers_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

    if ((clean_df['chanID'] == chanID) & (clean_df['unitID'] == unitID)).any():
        dest = f'{units_clean_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

# assert len(clean_df) == len(glob.glob(f'{units_clean_dir}/*.png')), f'Length mismatch: clean_df has {len(clean_df)} rows, but {len(glob.glob(f"{units_clean_dir}/*.png"))} files in {units_clean_dir}'
# assert len(maybes_df) == len(glob.glob(f'{units_maybes_dir}/*.png')), f'Length mismatch: maybes_df has {len(maybes_df)} rows, but {len(glob.glob(f"{units_maybes_dir}/*.png"))} files in {units_maybes_dir}'
# assert len(mergers_df) == len(glob.glob(f'{units_mergers_dir}/*.png')), f'Length mismatch: mergers_df has {len(mergers_df)} rows, but {len(glob.glob(f"{units_mergers_dir}/*.png"))} files in {units_mergers_dir}'

save = False # reset to be safe
possible_neurs_df

Copying to ../../data/units/clean/202521/A220_CL_2374_THM_1.png
Copying to ../../data/units/clean/202521/A217_CL_1148_THM_1.png
Copying to ../../data/units/clean/202521/A219_CL_1709_THM_1.png
Copying to ../../data/units/clean/202521/A220_CL_2305_THM_1.png
Copying to ../../data/units/clean/202521/A218_CL_3845_THM_1.png


,chanID,unitID,keep,merge_cluster,notes
0,217,1148,1.0,NaN,NaN
1,218,3845,1.0,NaN,NaN
2,218,3887,1.0,1.0,NaN
3,218,3888,1.0,1.0,NaN
4,219,1709,1.0,NaN,NaN
5,220,2305,1.0,NaN,NaN
6,220,2374,1.0,NaN,NaN
7,222,878,1.0,NaN,NaN
8,223,2489,1.0,NaN,NaN


### 3. create neur_df for neurs in possible_neurs_df and taking in spike times from osort mat files

### helpers

In [18]:
def getunitID2spikes(unitIDs, spikes, possible_neurs_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID not in possible_neurs_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [19]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, possible_neurs_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
neur_df = pd.merge(
    neur_df,
    possible_neurs_df[['chanID', 'unitID', 'keep', 'merge_cluster']],
    on=['chanID', 'unitID'],
    how='left'
)
neur_df = neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
assert len(neur_df) == len(possible_neurs_df), f'Length mismatch: neur_df has {len(neur_df)} rows, possible_neurs_df has {len(possible_neurs_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster
0,217,1148,"[34.2711, 34.276633333333336, 34.3298, 36.1802...",1067,0.644605,1.0,NaN
1,218,3845,"[186.2033, 189.2548666666667, 189.284366666666...",3452,2.174044,1.0,NaN
2,218,3887,"[0.34526666666666667, 0.3496, 0.35416666666666...",15705,8.859160,1.0,1.0
3,218,3888,"[0.3418666666666667, 0.6712, 1.758166666666666...",3976,2.241688,1.0,1.0
4,219,1709,"[5.8342, 33.16630000000001, 34.1727, 34.288, 3...",2786,1.606023,1.0,NaN
5,220,2305,"[34.1908, 34.4417, 34.516533333333335, 34.5623...",13423,7.790637,1.0,NaN
6,220,2374,"[10.1639, 32.283433333333335, 35.6210666666666...",710,0.406395,1.0,NaN
7,222,878,"[0.05560000000000001, 0.7493333333333334, 0.82...",3460,1.949670,1.0,NaN
8,223,2489,"[0.047733333333333336, 0.1006, 0.1926, 0.2007,...",2159,1.216535,1.0,NaN


### add region column by mapping channel -> region (label)

In [20]:
if patient != 22:
    chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

    # variable across patients
    if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
    elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

    labelMap = chanMap['LabelMap'].flatten() # contains region labels
    labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

    nan_mask = ~np.isnan(channelMap)
    channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

    neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))

else: neur_df['region'] = np.arange(len( neur_df)) # for pt22, just assign region = chanID since no chanMap available

neur_df


,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster,region
0,217,1148,"[34.2711, 34.276633333333336, 34.3298, 36.1802...",1067,0.644605,1.0,NaN,mRAHIP6
1,218,3845,"[186.2033, 189.2548666666667, 189.284366666666...",3452,2.174044,1.0,NaN,mRAHIP7
2,218,3887,"[0.34526666666666667, 0.3496, 0.35416666666666...",15705,8.859160,1.0,1.0,mRAHIP7
3,218,3888,"[0.3418666666666667, 0.6712, 1.758166666666666...",3976,2.241688,1.0,1.0,mRAHIP7
4,219,1709,"[5.8342, 33.16630000000001, 34.1727, 34.288, 3...",2786,1.606023,1.0,NaN,mRAHIP8
5,220,2305,"[34.1908, 34.4417, 34.516533333333335, 34.5623...",13423,7.790637,1.0,NaN,220
6,220,2374,"[10.1639, 32.283433333333335, 35.6210666666666...",710,0.406395,1.0,NaN,220
7,222,878,"[0.05560000000000001, 0.7493333333333334, 0.82...",3460,1.949670,1.0,NaN,222
8,223,2489,"[0.047733333333333336, 0.1006, 0.1926, 0.2007,...",2159,1.216535,1.0,NaN,223


### add coordinates columns by mapping region->coords

In [21]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

In [22]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
# ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?
# atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
# atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
if patient == 22:
    neur_df = neur_df.copy()
    neur_df[['x', 'y', 'z']] = 0
else:
    neur_df = (neur_df
                    .merge(region2id_df, on='region', how='left')
                    .merge(id2xyz_df, on='ID', how='left'))
                  #   .merge(xyz2atlasRegions, on='ID', how='left')
    neur_df = neur_df.drop(columns=['ID'])
    
# sort by chanID
neur_df = neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
neur_df

,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster,region,x,y,z
0,217,1148,"[34.2711, 34.276633333333336, 34.3298, 36.1802...",1067,0.644605,1.0,NaN,mRAHIP6,19.800004,-1.29937,-15.600003
1,218,3845,"[186.2033, 189.2548666666667, 189.284366666666...",3452,2.174044,1.0,NaN,mRAHIP7,18.600004,-1.29937,-15.600003
2,218,3887,"[0.34526666666666667, 0.3496, 0.35416666666666...",15705,8.859160,1.0,1.0,mRAHIP7,18.600004,-1.29937,-15.600003
3,218,3888,"[0.3418666666666667, 0.6712, 1.758166666666666...",3976,2.241688,1.0,1.0,mRAHIP7,18.600004,-1.29937,-15.600003
4,219,1709,"[5.8342, 33.16630000000001, 34.1727, 34.288, 3...",2786,1.606023,1.0,NaN,mRAHIP8,21.000004,-1.29937,-15.600003
5,220,2305,"[34.1908, 34.4417, 34.516533333333335, 34.5623...",13423,7.790637,1.0,NaN,220,NaN,NaN,NaN
6,220,2374,"[10.1639, 32.283433333333335, 35.6210666666666...",710,0.406395,1.0,NaN,220,NaN,NaN,NaN
7,222,878,"[0.05560000000000001, 0.7493333333333334, 0.82...",3460,1.949670,1.0,NaN,222,NaN,NaN,NaN
8,223,2489,"[0.047733333333333336, 0.1006, 0.1926, 0.2007,...",2159,1.216535,1.0,NaN,223,NaN,NaN,NaN


### 4. Merge clusters as needed

In [23]:
# merged_rows = []
# merge_mask = neur_df['merge_cluster'].notna()

# # extract merger rows as df
# for _, merge_group in neur_df[merge_mask].groupby('merge_cluster'):
#     merged_spikes = np.sort(np.concatenate(merge_group['spikes'].values))

#     # update row with merged info
#     first_row = merge_group.iloc[0]
#     merged_rows.append({
#         **{col: first_row[col] for col in neur_df.columns},
#         'unitID':     '/'.join(merge_group['unitID'].astype(str).tolist()),
#         'spikes':     merged_spikes,
#         'num_spikes': len(merged_spikes),
#         'FR':         len(merged_spikes) / (merged_spikes[-1] - merged_spikes[0]),
#         'keep':       1,
#         # chanID, region, merge_cluster, x, y, z already from iloc[0]
#     })

# # concat original df without merged rows + new merged rows
# neur_df = pd.concat(
#     [neur_df[~merge_mask], pd.DataFrame(merged_rows)],
#     ignore_index=True
# )
# neur_df = neur_df.sort_values(by=['chanID']).reset_index(drop=True)
# neur_df


### 5. inspect, typecast, & save

In [24]:
neur_df['spikes'] = neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays
print('example neuron')
print(f'last 5 spikes (s): {neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {neur_df["spikes"].iloc[0][-5:]/60}')

neur_df['unitID'] = neur_df['unitID'].astype(str)
 
# add patient as first column
if 'patient' not in neur_df: neur_df.insert(0, 'patient', patient)

# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if os.path.exists(parquet_path):     print(f'File already exists, skipping: {parquet_path}')
else:     neur_df.to_parquet(parquet_path, index=False)
neur_df.to_parquet(parquet_path, index=False)

example neuron
last 5 spikes (s): [1680.66743333 1683.6287     1685.4134     1688.2059     1689.5486    ]
last 5 spikes (min): [28.01112389 28.06047833 28.09022333 28.136765   28.15914333]
File already exists, skipping: ../../results/202521/records/processed_data/df_neurs.parquet
